In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip '/content/drive/MyDrive/M.L./Copy of hindi-visual-genome-11.zip' -d /content/hindi_visual_genome


Streaming output truncated to the last 5000 lines.
  inflating: /content/hindi_visual_genome/hindi-visual-genome-11/hindi-visual-genome-11/hindi-visual-genome-train.images/2403839.jpg  
  inflating: /content/hindi_visual_genome/hindi-visual-genome-11/hindi-visual-genome-11/hindi-visual-genome-train.images/2403843.jpg  
  inflating: /content/hindi_visual_genome/hindi-visual-genome-11/hindi-visual-genome-11/hindi-visual-genome-train.images/2403846.jpg  
  inflating: /content/hindi_visual_genome/hindi-visual-genome-11/hindi-visual-genome-11/hindi-visual-genome-train.images/2403849.jpg  
  inflating: /content/hindi_visual_genome/hindi-visual-genome-11/hindi-visual-genome-11/hindi-visual-genome-train.images/2403852.jpg  
  inflating: /content/hindi_visual_genome/hindi-visual-genome-11/hindi-visual-genome-11/hindi-visual-genome-train.images/2403856.jpg  
  inflating: /content/hindi_visual_genome/hindi-visual-genome-11/hindi-visual-genome-11/hindi-visual-genome-train.images/2403860.jpg  
  in

In [ ]:
import os
import pandas as pd
from PIL import Image
import numpy as np
import torch
from torchvision import transforms, models
from tqdm import tqdm

# === Paths ===
test_image_dir = "/content/hindi_visual_genome/hindi-visual-genome-11/hindi-visual-genome-11/hindi-visual-genome-test.images"
test_txt_path = "/content/hindi_visual_genome/hindi-visual-genome-11/hindi-visual-genome-11/hindi-visual-genome-test.txt"
output_feat_dir = "/content/TestSet_ROI_Features"
os.makedirs(output_feat_dir, exist_ok=True)

# === Load bounding boxes & captions ===
df = pd.read_csv(test_txt_path, sep='\t', header=None)
df.columns = ['image_id', 'x1', 'y1', 'w', 'h', 'en_caption', 'hi_caption']

# === ResNet Preprocessing ===
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# === Load ResNet-50 without final FC layer ===
resnet = models.resnet50(pretrained=True)
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])  # remove FC layer
resnet.eval().cuda()

# === Extract ResNet Features ===
feature_paths = []

for i, row in tqdm(df.iterrows(), total=len(df)):
    image_id = str(row['image_id'])
    img_path = os.path.join(test_image_dir, f"{image_id}.jpg")
    save_path = os.path.join(output_feat_dir, f"{image_id}_{i}.npy")

    try:
        if not os.path.exists(img_path):
            print(f"❌ Image not found: {img_path}")
            feature_paths.append(None)
            continue

        img = Image.open(img_path).convert('RGB')

        # Crop ROI
        x1, y1, w, h = int(row['x1']), int(row['y1']), int(row['w']), int(row['h'])
        cropped = img.crop((x1, y1, x1 + w, y1 + h))

        # Transform and extract feature
        tensor_img = transform(cropped).unsqueeze(0).cuda()
        with torch.no_grad():
            feat = resnet(tensor_img).squeeze().cpu().numpy()  # (2048,)
        np.save(save_path, feat)
        feature_paths.append(save_path)

    except Exception as e:
        print(f"⚠️ Skipping {image_id} due to error: {e}")
        feature_paths.append(None)

print(f"\n✅ Done! Total features extracted: {len([f for f in feature_paths if f is not None])}")


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 150MB/s]
100%|██████████| 1595/1595 [00:23<00:00, 67.20it/s]


✅ Done! Total features extracted: 1595


In [ ]:
import shutil

# === Define paths ===
folder_to_zip = "/content/TestSet_ROI_Features"
zip_file_path = "/content/TestSet_ROI_Features.zip"
drive_dest_path = "/content/drive/MyDrive/TestSet_ROI_Features.zip"

# === Create ZIP file ===
shutil.make_archive(base_name=zip_file_path.replace('.zip', ''), format='zip', root_dir=folder_to_zip)
print(f"✅ Zipped folder saved at: {zip_file_path}")

# === Copy to Drive ===
shutil.copy(zip_file_path, drive_dest_path)
print(f"✅ ZIP file copied to Drive: {drive_dest_path}")


✅ Zipped folder saved at: /content/TestSet_ROI_Features.zip
✅ ZIP file copied to Drive: /content/drive/MyDrive/TestSet_ROI_Features.zip


In [ ]:
import os
import pandas as pd
from PIL import Image
import numpy as np
import torch
from torchvision import transforms, models
from tqdm import tqdm

# === Paths ===
test_image_dir = "/content/hindi_visual_genome/hindi-visual-genome-11/hindi-visual-genome-11/hindi-visual-genome-dev.images"
test_txt_path = "/content/hindi_visual_genome/hindi-visual-genome-11/hindi-visual-genome-11/hindi-visual-genome-dev.txt"
output_feat_dir = "/content/ValSet_ROI_Features"
os.makedirs(output_feat_dir, exist_ok=True)

# === Load bounding boxes & captions ===
df = pd.read_csv(test_txt_path, sep='\t', header=None)
df.columns = ['image_id', 'x1', 'y1', 'w', 'h', 'en_caption', 'hi_caption']

# === ResNet Preprocessing ===
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# === Load ResNet-50 without final FC layer ===
resnet = models.resnet50(pretrained=True)
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])  # remove FC layer
resnet.eval().cuda()

# === Extract ResNet Features ===
feature_paths = []

for i, row in tqdm(df.iterrows(), total=len(df)):
    image_id = str(row['image_id'])
    img_path = os.path.join(test_image_dir, f"{image_id}.jpg")
    save_path = os.path.join(output_feat_dir, f"{image_id}_{i}.npy")

    try:
        if not os.path.exists(img_path):
            print(f"❌ Image not found: {img_path}")
            feature_paths.append(None)
            continue

        img = Image.open(img_path).convert('RGB')

        # Crop ROI
        x1, y1, w, h = int(row['x1']), int(row['y1']), int(row['w']), int(row['h'])
        cropped = img.crop((x1, y1, x1 + w, y1 + h))

        # Transform and extract feature
        tensor_img = transform(cropped).unsqueeze(0).cuda()
        with torch.no_grad():
            feat = resnet(tensor_img).squeeze().cpu().numpy()  # (2048,)
        np.save(save_path, feat)
        feature_paths.append(save_path)

    except Exception as e:
        print(f"⚠️ Skipping {image_id} due to error: {e}")
        feature_paths.append(None)

print(f"\n✅ Done! Total features extracted: {len([f for f in feature_paths if f is not None])}")


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 998/998 [00:13<00:00, 71.39it/s]


✅ Done! Total features extracted: 998


In [ ]:
import shutil

# === Define paths ===
folder_to_zip = "/content/ValSet_ROI_Features"
zip_file_path = "/content/ValSet_ROI_Features.zip"
drive_dest_path = "/content/drive/MyDrive/ValSet_ROI_Features.zip"

# === Create ZIP file ===
shutil.make_archive(base_name=zip_file_path.replace('.zip', ''), format='zip', root_dir=folder_to_zip)
print(f"✅ Zipped folder saved at: {zip_file_path}")

# === Copy to Drive ===
shutil.copy(zip_file_path, drive_dest_path)
print(f"✅ ZIP file copied to Drive: {drive_dest_path}")

✅ Zipped folder saved at: /content/ValSet_ROI_Features.zip
✅ ZIP file copied to Drive: /content/drive/MyDrive/ValSet_ROI_Features.zip


In [ ]:
# === Unzip the test image archive into Colab ===
!unzip '/content/drive/MyDrive/M.L./hindi-visual-genome-challenge-test-set.images.zip' -d /content/hvg_test_images


Archive:  /content/drive/MyDrive/M.L./hindi-visual-genome-challenge-test-set.images.zip
  inflating: /content/hvg_test_images/hindi-visual-genome-challenge-test-set.images/1067.jpg  
  inflating: /content/hvg_test_images/hindi-visual-genome-challenge-test-set.images/107903.jpg  
  inflating: /content/hvg_test_images/hindi-visual-genome-challenge-test-set.images/107932.jpg  
  inflating: /content/hvg_test_images/hindi-visual-genome-challenge-test-set.images/107937.jpg  
  inflating: /content/hvg_test_images/hindi-visual-genome-challenge-test-set.images/1084.jpg  
  inflating: /content/hvg_test_images/hindi-visual-genome-challenge-test-set.images/1122.jpg  
  inflating: /content/hvg_test_images/hindi-visual-genome-challenge-test-set.images/1140.jpg  
  inflating: /content/hvg_test_images/hindi-visual-genome-challenge-test-set.images/1159262.jpg  
  inflating: /content/hvg_test_images/hindi-visual-genome-challenge-test-set.images/1159278.jpg  
  inflating: /content/hvg_test_images/hindi-v

In [ ]:
import os
import pandas as pd
from PIL import Image
import numpy as np
import torch
from torchvision import transforms, models
from tqdm import tqdm

# === Paths ===
test_image_dir = "/content/hvg_test_images/hindi-visual-genome-challenge-test-set.images"
test_txt_path = "/content/hindi_visual_genome/hindi-visual-genome-11/hindi-visual-genome-11/hindi-visual-genome-challenge-test-set.txt"
output_feat_dir = "/content/Challenge_TestSet_ROI_Features"
os.makedirs(output_feat_dir, exist_ok=True)

# === Load bounding boxes & captions ===
df = pd.read_csv(test_txt_path, sep='\t', header=None)
df.columns = ['image_id', 'x1', 'y1', 'w', 'h', 'en_caption', 'hi_caption']

# === ResNet Preprocessing ===
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# === Load ResNet-50 without final FC layer ===
resnet = models.resnet50(pretrained=True)
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])  # remove FC layer
resnet.eval().cuda()

# === Extract ResNet Features ===
feature_paths = []

for i, row in tqdm(df.iterrows(), total=len(df)):
    image_id = str(row['image_id'])
    img_path = os.path.join(test_image_dir, f"{image_id}.jpg")
    save_path = os.path.join(output_feat_dir, f"{image_id}_{i}.npy")

    try:
        if not os.path.exists(img_path):
            print(f"❌ Image not found: {img_path}")
            feature_paths.append(None)
            continue

        img = Image.open(img_path).convert('RGB')

        # Crop ROI
        x1, y1, w, h = int(row['x1']), int(row['y1']), int(row['w']), int(row['h'])
        cropped = img.crop((x1, y1, x1 + w, y1 + h))

        # Transform and extract feature
        tensor_img = transform(cropped).unsqueeze(0).cuda()
        with torch.no_grad():
            feat = resnet(tensor_img).squeeze().cpu().numpy()  # (2048,)
        np.save(save_path, feat)
        feature_paths.append(save_path)

    except Exception as e:
        print(f"⚠️ Skipping {image_id} due to error: {e}")
        feature_paths.append(None)

print(f"\n✅ Done! Total features extracted: {len([f for f in feature_paths if f is not None])}")

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 1400/1400 [00:18<00:00, 76.59it/s]


✅ Done! Total features extracted: 1400


In [ ]:
import shutil

# === Define paths ===
folder_to_zip = "/content/Challenge_TestSet_ROI_Features"
zip_file_path = "/content/Challenge_TestSet_ROI_Features.zip"
drive_dest_path = "/content/drive/MyDrive/Challenge_TestSet_ROI_Features.zip"

# === Create ZIP file ===
shutil.make_archive(base_name=zip_file_path.replace('.zip', ''), format='zip', root_dir=folder_to_zip)
print(f"✅ Zipped folder saved at: {zip_file_path}")

# === Copy to Drive ===
shutil.copy(zip_file_path, drive_dest_path)
print(f"✅ ZIP file copied to Drive: {drive_dest_path}")

✅ Zipped folder saved at: /content/Challenge_TestSet_ROI_Features.zip
✅ ZIP file copied to Drive: /content/drive/MyDrive/Challenge_TestSet_ROI_Features.zip
